# Agentic AI — Maintenance Assistant

**Goal:** Build an LLM agent that routes user questions to the right tool:
- predict_failure: for questions about specific machine sensor readings
- search_docs: for questions about maintenance procedures and guidelines

**Pattern:** ReAct (Reason + Act) — agent reasons before each action
and continues until it has enough information to answer.

**Why this matters:** Unlike fixed pipelines, the agent adapts its approach based on the question — calling one tool, multiple tools, or no tool depending on what the user actually needs.

In [1]:
import requests
import json
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add parent directory to path to import RAG components
sys.path.append(str(Path('..').resolve()))

load_dotenv('../2_rag/.env')

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
API_BASE_URL = "http://127.0.0.1:8000"

print("Setup complete")
print(f"API URL: {API_BASE_URL}")
print(f"Gemini key loaded: {'yes' if GOOGLE_API_KEY else 'NO - check .env file'}")

Setup complete
API URL: http://127.0.0.1:8000
Gemini key loaded: yes


In [2]:
def predict_failure(
    machine_type: str,
    air_temperature: float,
    process_temperature: float,
    rotational_speed: float,
    torque: float,
    tool_wear: float
) -> dict:
    """
    Predicts machine failure probability from sensor readings.
    Call this when the user provides specific sensor values for a machine.

    Returns failure probability, prediction label, and risk level.
    """
    payload = {
        "type": machine_type.upper(),
        "air_temperature": air_temperature,
        "process_temperature": process_temperature,
        "rotational_speed": rotational_speed,
        "torque": torque,
        "tool_wear": tool_wear
    }

    try:
        response = requests.post(
            f"{API_BASE_URL}/predict",
            json=payload,
            timeout=10
        )
        response.raise_for_status()
        result = response.json()
        return {
            "status": "success",
            "failure_probability": result["failure_probability"],
            "prediction": result["prediction"],
            "risk_level": result["risk_level"],
            "interpretation": f"Machine has {result['failure_probability']*100:.1f}% failure probability — {result['risk_level']} risk"
        }
    except requests.exceptions.ConnectionError:
        return {"status": "error", "message": "API not reachable. Make sure FastAPI server is running on port 8000."}
    except Exception as e:
        return {"status": "error", "message": str(e)}

# Test it
result = predict_failure("L", 298.1, 308.6, 1363.0, 68.0, 220.0)
print(json.dumps(result, indent=2))

{
  "status": "success",
  "failure_probability": 1.0,
  "prediction": "FAILURE",
  "risk_level": "HIGH",
  "interpretation": "Machine has 100.0% failure probability \u2014 HIGH risk"
}


In [4]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load the FAISS index built in Session 4
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "../2_rag/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

def search_docs(query: str, k: int = 3) -> dict:
    """
    Searches maintenance documentation for relevant information.
    Call this when the user asks general questions about maintenance
    procedures, failure causes, equipment guidelines, or best practices.

    Returns relevant passages from the maintenance manual.
    """
    try:
        docs = vectorstore.similarity_search(query, k=k)

        if not docs:
            return {
                "status": "success",
                "results": [],
                "message": "No relevant documents found for this query."
            }

        results = []
        for i, doc in enumerate(docs):
            results.append({
                "rank": i + 1,
                "content": doc.page_content,
                "source": doc.metadata.get("source", "maintenance_manual")
            })

        return {
            "status": "success",
            "query": query,
            "results": results,
            "summary": f"Found {len(results)} relevant passages"
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}

# Test it
result = search_docs("Why are particles smaller than 5 microns considered more dangerous for hydraulic systems than larger particles?")
print(json.dumps(result, indent=2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
  "status": "success",
  "query": "Why are particles smaller than 5 microns considered more dangerous for hydraulic systems than larger particles?",
  "results": [
    {
      "rank": 1,
      "content": "Particles smaller than 5 microns are highly abrasive. If present in sufficient quantities, these \ninvisible 'silt' particles cause rapid wear, destroying hydraulic components. \n \n \nQuantifying Particle Contamination \n \nSome level of particle contamination is always present in hydraulic fluid, even in new fluid. It \nis the size and quantity of these particles that we are concerned with. The level of",
      "source": "docs/maintenance_guide.pdf"
    },
    {
      "rank": 2,
      "content": "CLEARANCE IN MICRONS \nGear pump 0.5 \u2013 5.0 \nVane pump 0.5 \u2013 10 \nPiston pump 0.5 \u2013 5.0 \nServo valve 1.0 \u2013 4.0 \nControl valve 0.5 \u2013 40 \nLinear actuator 50 - 250 \n \nParticles larger than a component's internal clearances are not necessarily dangerous. Particle

In [5]:
# Health check — agent will fail silently if tools are broken
api_health = requests.get(f"{API_BASE_URL}/health").json()
print(f"API health: {api_health}")

test_prediction = predict_failure("M", 298.1, 308.6, 1551.0, 42.8, 0.0)
print(f"Prediction tool: {test_prediction['status']}")

test_search = search_docs("tool wear replacement")
print(f"Search tool: {test_search['status']}")

print("\nAll tools operational ✅" if all(
    t['status'] == 'success'
    for t in [test_prediction, test_search]
) else "\n⚠️ Fix tool errors before proceeding")

API health: {'status': 'healthy', 'model_loaded': True}
Prediction tool: success
Search tool: success

All tools operational ✅
